
## 05_orchestrator — Streaming Pipeline

Runs the full GitHub streaming pipeline end to end.

Order:
  1. Simulate landing  → drop new hourly files into Bronze
  2. Auto Loader       → detect new files → write to Silver
  3. Gold aggregate    → re-aggregate Silver → update Gold
  4. Stream monitor    → health check all layers

Difference from Project 1 orchestrator:
  Project 1 → simple %run chain (batch)
  Project 2 → parameterised (which hours to land)
            → stream-aware (waits for Auto Loader to finish)
            → health check built in (not separate step)

Study Note:
  In production this notebook would be triggered by
  Databricks Workflows on a schedule:
  e.g. "run every hour, land the previous hour's file"
  That would create a true near-real-time pipeline.

In [0]:
%python

# Define config variables inline
BASE_PATH = "/Volumes/workspace/default/github_streaming"
BRONZE_PATH = f"{BASE_PATH}/bronze/events"
SCHEMA_PATH = f"{BASE_PATH}/schemas"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_to_silver"
SILVER_TABLE = "default.silver_github_events"
GITHUB_ARCHIVE_BASE = "https://data.gharchive.org"
PIPELINE = {
    "source_date": "2024-01-01",
    "write_mode": "append",
}

from datetime import datetime

# ── Orchestrator run config ───────────────────────────────────
# STUDY NOTE:
#   Separating RUN CONFIG from PIPELINE CONFIG is important.
#   Pipeline config = stable settings (paths, table names)
#   Run config      = per-execution settings (which hours, date)
#
#   This allows the orchestrator to be called with different
#   parameters without touching pipeline_config.py.
#   In production these would come from:
#   - Databricks Workflows job parameters
#   - A config file per environment (dev/staging/prod)

RUN_CONFIG = {
    "date":          PIPELINE["source_date"],
    "hours_to_land": ["5", "6"],   # ← new hours not yet landed
    "run_monitor":   True,
    "fail_on_lag":   True,         # ← stop pipeline if lag > 0 after run
}

# Pipeline run log
run_log = {
    "run_start":  None,
    "run_end":    None,
    "stages":     {},
    "status":     "RUNNING"
}

def log_stage(name, status, start, end, details=None):
    duration = round((end - start).total_seconds(), 2)
    run_log["stages"][name] = {
        "status":   status,
        "duration": f"{duration}s",
        "details":  details
    }
    icon = "✓" if status == "SUCCESS" else "✗"
    print(f"  {icon} {name:<30} {status:<10} {duration}s")
    if details:
        print(f"      → {details}")

print("Orchestrator initialised ✓")
print(f"Run config: {RUN_CONFIG}")

In [0]:
%python

# ── Stage 1: Land new hourly files ───────────────────────────
# STUDY NOTE:
#   We land ONLY the hours specified in RUN_CONFIG.
#   In production this would automatically calculate
#   which hour to land based on current time:
#   hour_to_land = datetime.now().hour - 1
#   Making the pipeline fully automatic.

import requests
import gzip
import shutil
import os
import tempfile

run_log["run_start"] = datetime.now()

print("=" * 55)
print(f"PIPELINE RUN STARTED: {run_log['run_start']}")
print("=" * 55)

stage  = "01_simulate_landing"
start  = datetime.now()
landed = []

try:
    for hour in RUN_CONFIG["hours_to_land"]:
        url = f"{GITHUB_ARCHIVE_BASE}/{RUN_CONFIG['date']}-{hour}.json.gz"

        tmp_dir   = tempfile.mkdtemp()
        gz_path   = os.path.join(tmp_dir, f"{RUN_CONFIG['date']}-{hour}.json.gz")
        json_path = os.path.join(tmp_dir, f"{RUN_CONFIG['date']}-{hour}.json")

        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(gz_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

        with gzip.open(gz_path, "rb") as f_in:
            with open(json_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)

        bronze_file = f"{BRONZE_PATH}/{RUN_CONFIG['date']}-{hour}.json"
        shutil.copy(json_path, bronze_file)
        shutil.rmtree(tmp_dir)
        landed.append(hour)

    log_stage(
        stage, "SUCCESS", start, datetime.now(),
        f"Landed hours: {landed}"
    )

except Exception as e:
    log_stage(stage, "FAILED", start, datetime.now(), str(e))
    raise

In [0]:
%python

# ── Stage 2: Auto Loader → Silver ────────────────────────────
# STUDY NOTE:
#   We call Auto Loader programmatically here —
#   not via %run — because we need to capture
#   stream metrics (rows added, files processed)
#   for the run log.
#
#   query.recentProgress gives stream metrics:
#   - numInputRows     → rows processed this batch
#   - inputRowsPerSecond → throughput
#   - numFiles         → files processed
#
#   This is how you monitor streams programmatically
#   in production — inspect recentProgress after each batch.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType

stage = "02_autoloader_silver"
start = datetime.now()

try:
    actor_schema = StructType([
        StructField("id",            LongType(),   True),
        StructField("login",         StringType(), True),
        StructField("display_login", StringType(), True),
        StructField("url",           StringType(), True),
    ])
    repo_schema = StructType([
        StructField("id",   LongType(),   True),
        StructField("name", StringType(), True),
        StructField("url",  StringType(), True),
    ])
    bronze_schema = StructType([
        StructField("id",         StringType(), True),
        StructField("type",       StringType(), True),
        StructField("actor",      actor_schema, True),
        StructField("repo",       repo_schema,  True),
        StructField("created_at", StringType(), True),
        StructField("public",     StringType(), True),
    ])

    df_stream = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", SCHEMA_PATH)
            .option("cloudFiles.inferColumnTypes", "true")
            .option("multiLine", "false")
            .schema(bronze_schema)
            .load(BRONZE_PATH)
    )

    df_silver_stream = df_stream.select(
        F.col("id")                     .alias("event_id"),
        F.col("type")                   .alias("event_type"),
        F.col("actor.id")               .alias("actor_id"),
        F.col("actor.login")            .alias("actor_login"),
        F.col("repo.id")                .alias("repo_id"),
        F.col("repo.name")              .alias("repo_name"),
        F.to_timestamp("created_at")    .alias("event_timestamp"),
        F.to_date("created_at")         .alias("event_date"),
        F.col("public")                 .alias("is_public"),
        F.col("_metadata.file_name")    .alias("source_file"),
        F.current_timestamp()           .alias("ingestion_timestamp"),
    )

    query = (
        df_silver_stream.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", CHECKPOINT_PATH)
            .trigger(availableNow=True)
            .toTable(SILVER_TABLE)
    )

    query.awaitTermination()

    # Debug: Display stream progress
    print("Stream recent progress:")
    for m in query.recentProgress:
        print(m)

    # Capture stream metrics
    metrics    = query.recentProgress
    rows_added = sum([(m.get("numInputRows") or 0) for m in metrics])
    batches    = len(metrics)

    log_stage(
        stage, "SUCCESS", start, datetime.now(),
        f"Rows added: {rows_added:,} across {batches} batch(es)"
    )

except Exception as e:
    log_stage(stage, "FAILED", start, datetime.now(), str(e))
    raise

In [0]:
%python

# ── Stage 3: Re-aggregate Gold ────────────────────────────────
# STUDY NOTE:
#   Every time Silver grows, Gold needs to be updated.
#   We re-run all 3 Gold aggregations and MERGE results.
#   MERGE ensures only changed rows are updated —
#   efficient even when Silver has millions of rows.

from delta.tables import DeltaTable

GOLD_SUMMARY_TABLE = "default.gold_github_summary"
GOLD_REPO_TABLE    = "default.gold_repo_activity"
GOLD_ACTOR_TABLE   = "default.gold_actor_activity"

stage = "03_gold_aggregate"
start = datetime.now()

try:
    df_silver = spark.table(SILVER_TABLE)

    # Summary aggregation
    df_summary = df_silver.groupBy("event_date", "event_type").agg(
        F.count("*")                            .alias("total_events"),
        F.approx_count_distinct("actor_login")  .alias("unique_actors"),
        F.approx_count_distinct("repo_name")    .alias("unique_repos"),
        F.current_timestamp()                   .alias("last_updated")
    )

    # Repo aggregation
    df_repo = df_silver.filter(F.col("repo_name").isNotNull()) \
        .groupBy("repo_name", "event_date").agg(
            F.count("*")                            .alias("total_events"),
            F.approx_count_distinct("actor_login")  .alias("unique_contributors"),
            F.collect_set("event_type")             .alias("event_types"),
            F.current_timestamp()                   .alias("last_updated")
        )

    # Actor aggregation
    df_actor = df_silver.filter(F.col("actor_login").isNotNull()) \
        .groupBy("actor_login", "event_date").agg(
            F.count("*")                            .alias("total_events"),
            F.approx_count_distinct("repo_name")    .alias("repos_touched"),
            F.sum(F.when(F.col("event_type") == "PushEvent", 1).otherwise(0)).alias("push_count"),
            F.sum(F.when(F.col("event_type") == "PullRequestEvent", 1).otherwise(0)).alias("pr_count"),
            F.sum(F.when(F.col("event_type") == "WatchEvent", 1).otherwise(0)).alias("star_count"),
            F.current_timestamp()                   .alias("last_updated")
        )

    def merge_gold(df, table, keys):
        if not spark.catalog.tableExists(table):
            df.write.format("delta").mode("overwrite").saveAsTable(table)
        else:
            cond = " AND ".join([f"target.{k} = source.{k}" for k in keys])
            DeltaTable.forName(spark, table).alias("target").merge(
                df.alias("source"), cond
            ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    merge_gold(df_summary, GOLD_SUMMARY_TABLE, ["event_date", "event_type"])
    merge_gold(df_repo,    GOLD_REPO_TABLE,    ["repo_name", "event_date"])
    merge_gold(df_actor,   GOLD_ACTOR_TABLE,   ["actor_login", "event_date"])

    total_gold = (
        spark.table(GOLD_SUMMARY_TABLE).count() +
        spark.table(GOLD_REPO_TABLE).count() +
        spark.table(GOLD_ACTOR_TABLE).count()
    )

    log_stage(
        stage, "SUCCESS", start, datetime.now(),
        f"Total Gold rows: {total_gold:,}"
    )

except Exception as e:
    log_stage(stage, "FAILED", start, datetime.now(), str(e))
    raise

In [0]:
%python

# ── Stage 4: Pipeline health check ───────────────────────────
# STUDY NOTE:
#   Running a health check at the END of the pipeline
#   is called a "post-run assertion" or "pipeline gate".
#   If the gate fails, the pipeline is marked as failed
#   even if all stages technically completed.
#   This catches silent failures — e.g. stream ran but
#   wrote 0 rows (no error thrown, but also no data).

stage = "04_stream_monitor"
start = datetime.now()

try:
    # Lag check
    bronze_files    = [f.name for f in dbutils.fs.ls(BRONZE_PATH) if f.name.endswith(".json")]
    df_silver       = spark.table(SILVER_TABLE)
    processed       = [r.source_file for r in df_silver.select("source_file").distinct().collect()]
    pending         = [f for f in bronze_files if f not in processed]
    silver_count    = df_silver.count()

    if RUN_CONFIG["fail_on_lag"] and len(pending) > 0:
        raise Exception(f"LAG DETECTED: {len(pending)} unprocessed files: {pending}")

    log_stage(
        stage, "SUCCESS", start, datetime.now(),
        f"Silver: {silver_count:,} rows | Lag: {len(pending)} files"
    )

except Exception as e:
    log_stage(stage, "FAILED", start, datetime.now(), str(e))
    raise

In [0]:
%python

# ── Pipeline Run Summary ──────────────────────────────────────
run_log["run_end"] = datetime.now()
run_log["status"]  = "SUCCESS"

total_duration = round(
    (run_log["run_end"] - run_log["run_start"]).total_seconds(), 2
)

print("\n" + "=" * 55)
print("STREAMING PIPELINE RUN SUMMARY")
print("=" * 55)
print(f"Started   : {run_log['run_start']}")
print(f"Finished  : {run_log['run_end']}")
print(f"Duration  : {total_duration}s")
print("-" * 55)
print(f"{'Stage':<32} {'Status':<10} {'Duration'}")
print("-" * 55)

for name, info in run_log["stages"].items():
    icon = "✓" if info["status"] == "SUCCESS" else "✗"
    print(f"{icon} {name:<31} {info['status']:<10} {info['duration']}")
    if info["details"]:
        print(f"    → {info['details']}")

print("=" * 55)

# Final layer counts
print("\nFinal Layer Counts:")
print(f"  Bronze files : {len(bronze_files)}")
print(f"  Silver events: {spark.table(SILVER_TABLE).count():,}")
print(f"  Gold Summary : {spark.table(GOLD_SUMMARY_TABLE).count():,}")
print(f"  Gold Repos   : {spark.table(GOLD_REPO_TABLE).count():,}")
print(f"  Gold Actors  : {spark.table(GOLD_ACTOR_TABLE).count():,}")

print("=" * 55)
print(f"STATUS: {run_log['status']} ✓")
print("=" * 55)